# TDI 101523 — L&H (Life & Health) DQ Check

## Purpose
Checks data availability for all source tables and columns used in the TDI 101523 L&H metric calculations.

## Pattern
Follows the TDW PT 101015 DQ check approach: one check per (table, column), recording which CDEs use each column.

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

In [ ]:
from datetime import date

# DQ results table
TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '101523'
lob_desc = 'TDI - Life & Health'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

## Segment Data Sources

In [ ]:
# ============================================================
# Source: p_parsed_2025_new.marketing_lh_customers_combined
# Main L&H customer/policy table
# ============================================================
print('p_parsed_2025_new.marketing_lh_customers_combined')
print('=' * 60)

check_column('SD1,SD3,1.6,1.7,1.1,1.2,1.3,1.4,1.5,SD6,ABAC1.2', 'Segment/Centralized', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'first_name')

check_column('SD1,SD3,1.6,1.7,1.1,1.2,1.3,1.4,1.5,SD6,ABAC1.2', 'Segment/Centralized', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'last_name')

check_column('SD1,SD3,1.6,1.7,1.1,1.2,1.3,1.4,1.5,SD6,ABAC1.2', 'Segment/Centralized', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'dob')

check_column('1.1,1.2,1.3,1.4,1.5,1.8,1.9,SD6,ABAC1.2', 'Segment/Centralized', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'customer_number')

check_column('SD3,1.6,1.7', 'Segment', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'policy_number')

check_column('SD1,SD3,1.6,1.7,6.5B', 'Segment', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'month_end')

check_column('1.1,1.2,1.3,1.4,1.5,1.8,1.9,3.17,3.18', 'Centralized', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'datasource')

In [ ]:
# ============================================================
# Source: p_parsed_2025_new.marketing_lh_customers_combined
# Country column (derived via policy_number join)
# Note: country is obtained through CTE ck_t joining on policy_number
# ============================================================
print('p_parsed_2025_new.marketing_lh_customers_combined — country availability')
print('=' * 60)

# Check country availability via the policy-level view
# The CTE does: select policy_number, max(country) ck from marketing_lh_customers_combined group by 1
check_column('SD3,1.6,1.7', 'Segment', 'Rahona',
             'p_parsed_2025_new.marketing_lh_customers_combined', 'country')

## Policy / Enrollment Data Sources (4.1A, 4.1B)

In [ ]:
# ============================================================
# Source: p_parsed_2025_new.cp_new_enrollment_dates
# New enrollment dates for 4.1A/4.1B
# ============================================================
print('p_parsed_2025_new.cp_new_enrollment_dates')
print('=' * 60)

check_column('4.1A,4.1B', 'Segment', 'Rahona',
             'p_parsed_2025_new.cp_new_enrollment_dates', 'policy_number')

In [ ]:
# ============================================================
# Source: p_parsed_2025_new.bpi_new_enrollment_dates
# BPI enrollment dates for 4.1A/4.1B
# ============================================================
print('p_parsed_2025_new.bpi_new_enrollment_dates')
print('=' * 60)

check_column('4.1A,4.1B', 'Segment', 'Rahona',
             'p_parsed_2025_new.bpi_new_enrollment_dates', 'policy_number')

## Reference / Lookup Tables

In [ ]:
# ============================================================
# Source: p_parsed_2025_new.sanctions_country_risk_rating_2025_all
# Country risk ratings for 1.6 (uses different sanctions table than TDGI)
# ============================================================
print('p_parsed_2025_new.sanctions_country_risk_rating_2025_all')
print('=' * 60)

check_column('1.6,1.7', 'Reference', 'Rahona',
             'p_parsed_2025_new.sanctions_country_risk_rating_2025_all', 'country')

check_column('1.6', 'Reference', 'Rahona',
             'p_parsed_2025_new.sanctions_country_risk_rating_2025_all', 'RiskRating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.sanctions_country_risk_rating_2025
# Used for branch risk rating 5.6, 5.7
# ============================================================
print('ra_adido_2025.sanctions_country_risk_rating_2025')
print('=' * 60)

check_column('5.6,5.7', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'country')

check_column('5.6,5.7', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'RiskRating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.country_ref_list_ca2025
# Country reference for branch risk 5.2-5.5
# ============================================================
print('ra_adido_2025.country_ref_list_ca2025')
print('=' * 60)

check_column('5.2,5.3,5.4,5.5', 'Reference', 'ADIDO',
             'ra_adido_2025.country_ref_list_ca2025', 'Country_Name')

check_column('5.2,5.3,5.4,5.5', 'Reference', 'ADIDO',
             'ra_adido_2025.country_ref_list_ca2025', '2025_Risk_Rating')

## Branch Data

In [ ]:
# ============================================================
# Source: ra_fy25_view.td_insurance_branches_2025
# Branch data for 5.1-5.9 (L&H uses life_health_ins_ind)
# ============================================================
print('ra_fy25_view.td_insurance_branches_2025')
print('=' * 60)

check_column('5.1,5.9', 'Branch', 'ADIDO',
             'ra_fy25_view.td_insurance_branches_2025', 'life_health_ins_ind')

check_column('5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Branch', 'ADIDO',
             'ra_fy25_view.td_insurance_branches_2025', 'country')

## L&H Custom Metrics (6.5A)

In [ ]:
# ============================================================
# Source: ra_fy24_view.tdlh_custom_metrics_2024
# Custom volume metrics for 6.5A
# ============================================================
print('ra_fy24_view.tdlh_custom_metrics_2024')
print('=' * 60)

check_column('6.5A', 'Segment', 'Rahona',
             'ra_fy24_view.tdlh_custom_metrics_2024', 'lch_volume_cnt')

check_column('6.5A', 'Segment', 'Rahona',
             'ra_fy24_view.tdlh_custom_metrics_2024', 'lob_channel_desc')

check_column('6.5A', 'Segment', 'Rahona',
             'ra_fy24_view.tdlh_custom_metrics_2024', 'lob_type_desc')

## Transaction Data

In [ ]:
# ============================================================
# Source: ra_fy25_view.tdi_rpt_main3_history_view
# Transaction data for 3.1-3.16
# ============================================================
print('ra_fy25_view.tdi_rpt_main3_history_view')
print('=' * 60)

check_column('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona',
             'ra_fy25_view.tdi_rpt_main3_history_view', 'postdate')

check_column('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona',
             'ra_fy25_view.tdi_rpt_main3_history_view', 'endpt')

check_column('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona',
             'ra_fy25_view.tdi_rpt_main3_history_view', 'curtype')

check_column('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona',
             'ra_fy25_view.tdi_rpt_main3_history_view', 'curtime_amount')

## Centralized / Sanctions / STR / UTR

In [ ]:
# ============================================================
# Source: rafy2025_centralized.crv_scorable_cust_ca
# CRV scoring for 1.1 (unscored customers)
# ============================================================
print('rafy2025_centralized.crv_scorable_cust_ca')
print('=' * 60)

check_column('1.1', 'Centralized', 'Rahona',
             'rafy2025_centralized.crv_scorable_cust_ca', 'CUST_INTRL_ID')

check_column('1.1', 'Centralized', 'Rahona',
             'rafy2025_centralized.crv_scorable_cust_ca', 'v_entity_id')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.customer_scorable_rated_ca
# Risk-rated customers for 1.2, 1.3, 1.4
# ============================================================
print('rafy2025_centralized.customer_scorable_rated_ca')
print('=' * 60)

check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_rated_ca', 'CUST_INTRL_ID')

check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_rated_ca', 'risk_rating')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.customer_scorable_unrated_ca
# Unrated customers for 1.5
# ============================================================
print('rafy2025_centralized.customer_scorable_unrated_ca')
print('=' * 60)

check_column('1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_unrated_ca', 'CUST_INTRL_ID')

In [ ]:
# ============================================================
# Source: ra_adido_2025.true_sanction_match_2025
# True sanctions matching for 1.8
# Note: L&H uses 'true_sanction_match_2025' (no 's')
# ============================================================
print('ra_adido_2025.true_sanction_match_2025')
print('=' * 60)

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'Customername')

In [ ]:
# ============================================================
# Source: ra_adido_2025.customer_sanctioned_blocked_property_2025
# Blocked property for 1.9
# ============================================================
print('ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)

check_column('1.9', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.CDE_SD_6_1yr_fy2025
# New customers (<1yr) for SD6
# ============================================================
print('rafy2025_centralized.CDE_SD_6_1yr_fy2025')
print('=' * 60)

check_column('SD6', 'Centralized', 'Rahona',
             'rafy2025_centralized.CDE_SD_6_1yr_fy2025', 'full_nm')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details
# Unusual Transaction Referrals for 3.17
# ============================================================
print('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details')
print('=' * 60)

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_nm')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.TD_SAR_CDE_3_18_2025
# SARs/STRs for 3.18
# ============================================================
print('rafy2025_centralized.TD_SAR_CDE_3_18_2025')
print('=' * 60)

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_Internal_Unique_ID')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_LEILAs_Customer_Nam')

## ABAC Data Sources

In [ ]:
# ============================================================
# Source: ra_adido_2025.pep_list_2025_exploded
# PEP list for ABAC1.2
# ============================================================
print('ra_adido_2025.pep_list_2025_exploded')
print('=' * 60)

check_column('ABAC1.2', 'ABAC', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'ENTITY')

check_column('ABAC1.2', 'ABAC', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'CUST_INTRL_ID')

In [ ]:
# ============================================================
# Source: ra_adido_2025.TD_Country_Risk_Rating_ABAC
# ABAC country risk for ABAC5.1
# ============================================================
print('ra_adido_2025.TD_Country_Risk_Rating_ABAC')
print('=' * 60)

check_column('ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'DerivedDesc')

check_column('ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating')

## Summary

In [ ]:
# ============================================================
# Display all DQ results for this AU
# ============================================================
results = spark.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE lob_id = '{lob_id}'
      AND run_date = '{today_date}'
    ORDER BY src_table_name, data_element
""")

print(f'Total DQ checks: {results.count()}')
display(results)